In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone

In [2]:
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

index = pc.Index("pinecone-data")

In [3]:
def retrieve(query, top_k=3):

    query_embedding = client.embeddings.create(
        input=query,
        model="text-embedding-3-small"
    ).data[0].embedding

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True,
        namespace="youtube_rag_dataset"
    )

    retrieved_docs = []
    sources = []

    for match in results["matches"]:
        retrieved_docs.append(match["metadata"]["text"])
        sources.append(match["metadata"]["title"])

    return retrieved_docs, sources

In [4]:
def build_prompt(query, docs):

    prompt = f"""
Use the context below to answer the user's question.

Context:
{chr(10).join(docs)}

Question:
{query}

Answer:
"""

    return prompt

In [5]:
def question_answering(prompt):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content

In [6]:
query = "What is Retrieval Augmented Generation?"

docs, sources = retrieve(query)

prompt = build_prompt(query, docs)

answer = question_answering(prompt)

print(answer)

Retrieval Augmented Generation (RAG) is a technique that combines retrieval-based methods with generative models to enhance the quality and relevance of generated text. In this approach, a model first retrieves relevant documents or information from a large database or knowledge base based on a given query. This information is then used as context to generate a more informed and contextually accurate response.

The process typically involves two main components: 

1. **Retriever**: This component searches and retrieves relevant documents or snippets from a knowledge base, such as a document store indexed with embeddings. The retrieval is often based on semantic similarity, allowing the model to fetch contextually relevant information.

2. **Generator**: Once the relevant documents are retrieved, a generative model (like GPT or another language model) processes this information to produce a coherent and context-aware response or text output.

By integrating retrieval into the generation

In [7]:
print("Sources:")

for source in sources:
    print("-", source)

Sources:
- Q&A Document Retrieval With DPR
- Q&A Document Retrieval With DPR
- Making The Most of Data: Augmented SBERT
